<h1>Note: This is a simple introduction to einstein python library, that I might use on forward to explain the General Relativity concepts using Python.</h1>

For the official documentation of Einstein Library, Visit: https://einsteinpy-einsteinpy.readthedocs.io/en/latest/index.html

<h1>Install the Library on Colab</h1>

In [1]:
!pip install einsteinpy

- Make a 'Coordinate System' list

We can make our grid lines / Coordinate system using undefined symbols using Sympy, and then create a list storing them.

In [2]:
import sympy as sp

# Define raw algebraic symbols for a 2D Polar Grid (Radius and Angle)
r, theta = sp.symbols('r theta') # Index 0 means the radial distance (r), and Index 1 means the angular swing (\theta).

# Bundle them into a list
polar_coords = [r, theta]

- MetricTensor
    - Creates a Metric Tensor using a raw matrix and the coordinate system

Now that we have our grid lines, we need our correction table. As you walk around a polar grid, stepping outward costs a flat $(1)$, but swinging around a circle scales by your distance squared $(r^{2})$!

$\left(\begin{matrix}g_{00}&g_{01}\\ g_{10}&g_{11}\end{matrix}\right)\rightarrow \left(\begin{matrix}g_{rr}&g_{r\theta }\\ g_{\theta r}&g_{\theta \theta }\end{matrix}\right)\rightarrow \left(\begin{matrix}1&0\\ 0&r^{2}\end{matrix}\right)$

In [3]:
from einsteinpy.symbolic import MetricTensor

# Create a raw 2D diagonal Polar matrix using standard SymPy
g_matrix = sp.diag(1, r**2) # These are the multipliers of our coordinate system

# Wrap it into EinsteinPy's MetricTensor object along with our coordinates
metric = MetricTensor(g_matrix.tolist(), polar_coords) # Converts the raw matrix to a list and takes it, and then the coordinate system list

# MetricTensor(g_matrix.tolist(), polar_coords) basically glues 1 from the diagonal matrix to the r coordinate, and r**2 to the theta coordinate
# Giving the MetricTensor polar_coords is necessary because it dictates the exact characters the computer is forced to search for inside your matrix.


# View the underlying tensor matrix
print(metric.tensor()) # Output the actual metric tensor
print() # Newline

print(metric[0][0]) # This is g_11 = (1) - (rr) component. Page 38
print(metric[0][1]) # This is g_12 = (0) - (r \theta) component. Page 38
print(metric[1][0]) # This is g_21 = (0) - (\theta r) component. Page 38
print(metric[1][1]) # This is g_22 = (r^2) - (\theta \theta) component. Page 38

[[1, 0], [0, r**2]]

1
0
0
r**2


- ChristoffelSymbols
    - Creates Christoffel symbols from a metric tensor


    - NOTE: Remember the core rule: Christoffel symbols measure how your coordinate grid lines bend relative to local metric units.


<br>
<br>

Instead of calculating partial derivatives $(\partial _{\mu }g_{\alpha \beta })$ by hand, we pass our metric object directly to ChristoffelSymbols

In [4]:
from einsteinpy.symbolic import ChristoffelSymbols

# Feed the metric into the automated connection solver
christoffel = ChristoffelSymbols.from_metric(metric)

# Extract a specific component: Gamma^r_{theta, theta}
# Remember: Python indices start at 0 (0 = r, 1 = theta)
# This asks: "If moving in theta, how much does the theta-basis vector tilt along r?"
gamma_1_22 = christoffel[0, 1, 1] # - Translates to {\gamma^r_theta_theta}

gamma_1_11 = christoffel[0,0,0]
gamma_1_12 = christoffel[0,0,1]
gamma_1_21 = christoffel[0,1,0]

gamma_2_11 = christoffel[1, 0, 0]
gamma_2_12 = christoffel[1, 0, 1]
gamma_2_21 = christoffel[1, 1, 0]
gamma_2_22 = christoffel[1, 1, 1]

print(f"The inward geometric tilt factor Gamma^r_theta_theta is: {gamma_1_22}")

# Other as well
print(gamma_1_11)
print(gamma_1_12)
print(gamma_1_21)
print()
print(gamma_2_11)
print(gamma_2_12)
print(gamma_2_21)
print(gamma_2_22)


The inward geometric tilt factor Gamma^r_theta_theta is: -r
0
0
0

0
1/r
1/r
0



| Index Pattern | Tensor Component | Value (Polar) |
| :--- | :--- | :--- |
| `christoffel[0, 1, 1]` | $\Gamma^{r}_{\theta\theta}$ | $-r$ |
| `christoffel[1, 0, 1]` | $\Gamma^{\theta}_{r\theta}$ | $1/r$ |
| `christoffel[1, 1, 0]` | $\Gamma^{\theta}_{\theta r}$ | $1/r$ |

*Table: Christoffel symbol indexing pattern: christoffel[upper, lower1, lower2]*
